# VAE Evaluation (Variational Autoencoder)

<div style="text-align: justify">

The following notebook evaluates the trained <b>Variational Autoencoder (VAE)</b> for the <b>Tau Anomaly Detection</b> analysis. It loads the trained checkpoint, computes per-event anomaly scores, and produces evaluation metrics (ROC AUC, SIC) and diagnostic visualisations including reconstruction quality, per-feature importance, and comprehensive VAE latent space analysis with posterior collapse monitoring.

</div>

## Pipeline Summary

| Step | Module | Description |
|------|--------|-------------|
| Config | `hydra.compose` | Load analysis and model configuration |
| DataModule | `datamodule.AnomalyDataModule` | Read mc.parquet, split bkg/sig, fit scaler, build dataloaders |
| Load | `vae.VariationalAutoencoder.load_from_checkpoint` | Load trained VAE checkpoint |
| Scores | `anomaly.reconstruction_error` | Compute per-event anomaly scores on predict set |
| Threshold | `anomaly.compute_threshold` | Percentile-based anomaly threshold on background |
| Metrics | `evaluation.compute_metrics` | ROC AUC, SIC, per-sample-type ROC |
| Plots | `plots` | Reconstruction error, ROC, SIC, per-feature importance, latent diagnostics |
| Save | `io.save_dataframe`, `json.dump` | Persist scores parquet and metrics JSON |

The same pipeline is available as a CLI via `uv run python run.py stage=evaluate model=vae`.

## Initialization

### Libraries

Configuration:
* [Hydra](https://hydra.cc/)
* [OmegaConf](https://omegaconf.readthedocs.io/)
* [pyrootutils](https://github.com/ashleve/pyrootutils)

Machine Learning:
* [PyTorch](https://pytorch.org/)
* [PyTorch Lightning](https://lightning.ai/docs/pytorch/stable/)
* [scikit-learn](https://scikit-learn.org/)

Visualization:
* [matplotlib](https://matplotlib.org/)

### Notebook

Activating autoreload of imported modules.

In [ ]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = "retina"

Initializing the project root.

In [ ]:
import pyrootutils

path = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)

Suppressing unessential warnings and applying ATLAS style.

In [ ]:
from src.utils import suppress_warnings
from src.visualization.plots import apply_atlas_style

suppress_warnings()
apply_atlas_style()

## Configuration

Loading the Hydra configuration with the VAE model config. All path parameters must match those used during training.

In [ ]:
from hydra import compose, initialize_config_dir

initialize_config_dir(config_dir=str(path / "configs"), version_base="1.3")
cfg = compose(config_name="config", overrides=["model=vae"])

Resolving input and output directories from config.

In [ ]:
from src.processing.analysis import get_output_paths

output_paths = get_output_paths(cfg)
dataframes_dir = path / output_paths["dataframes_dir"]
models_dir = path / output_paths["models_dir"]
plots_dir = path / output_paths["plots_dir"] / "vae_evaluation"

plots_dir.mkdir(parents=True, exist_ok=True)

## DataModule

Setting up the `AnomalyDataModule`. It reads the MC parquet, separates background from signal, fits a scaler on the background training split, and builds the predict set (background validation + all signal).

In [ ]:
import lightning as L

L.seed_everything(cfg.seed, workers=True)

In [ ]:
from src.models.datamodule import AnomalyDataModule

background_origins = {
    s["id"]
    for s in cfg.samples.background.samples
    if s["id"] not in set(cfg.samples.background.get("exclude", []))
}
print(f"Background origins: {background_origins}")

dm = AnomalyDataModule(
    mc_path=str(dataframes_dir / "mc.parquet"),
    background_origins=background_origins,
    normalization=cfg.model.normalization,
    val_fraction=cfg.pipeline.val_fraction,
    batch_size=cfg.model.batch_size,
    seed=cfg.seed,
)
dm.setup()
print(f"Features: {dm.n_features}")
print(f"Feature names: {dm.feature_names_}")

Gathering original features from the predict set.

In [ ]:
import torch

x_orig = torch.cat([batch[0] for batch in dm.predict_dataloader()])
print(
    f"Predict set: {len(x_orig):,} events "
    f"({int((dm.predict_labels == 0).sum()):,} bkg, "
    f"{int((dm.predict_labels == 1).sum()):,} sig)"
)

## Deserialization

### Load Checkpoint

Loading the trained Variational Autoencoder from checkpoint.

In [ ]:
from omegaconf import OmegaConf

from src.models.config import VAEConfig
from src.models.vae import VariationalAutoencoder

vae_params = dict(OmegaConf.to_container(cfg.model, resolve=True))
vae_cfg = VAEConfig(**vae_params)
model = VariationalAutoencoder.load_from_checkpoint(
    models_dir / "vae.ckpt", cfg=vae_cfg, n_features=dm.n_features
)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"VAE: {n_params:,} parameters")

### Anomaly Scores

Computing per-event reconstruction error on the predict set. Also extracting `mu` and `logvar` for latent space analysis.

In [ ]:
import numpy as np

from src.models.anomaly import (
    build_scores_frame,
    compute_threshold,
    per_feature_error,
    reconstruction_error,
)

x_hat_list = []
mu_list, logvar_list = [], []

with torch.no_grad():
    for batch in dm.predict_dataloader():
        x, _w = batch
        x_hat, mu, logvar = model(x)
        x_hat_list.append(x_hat)
        mu_list.append(mu)
        logvar_list.append(logvar)

x_hat = torch.cat(x_hat_list)
mu_np = torch.cat(mu_list).numpy()
logvar_np = torch.cat(logvar_list).numpy()
scores = reconstruction_error(x_orig, x_hat).numpy()

print(f"VAE scores: {len(scores):,} events")
print(f"  Background mean: {scores[dm.predict_labels == 0].mean():.6f}")
print(f"  Signal mean:     {scores[dm.predict_labels == 1].mean():.6f}")
print(f"  Latent dim: {mu_np.shape[1]}")

### Threshold

In [ ]:
bkg_scores = scores[dm.predict_labels == 0]
sig_scores = scores[dm.predict_labels == 1]

threshold = compute_threshold(
    bkg_scores,
    strategy=cfg.pipeline.threshold_strategy,
    percentile=cfg.pipeline.threshold_percentile,
)
print(f"Threshold ({cfg.pipeline.threshold_strategy}): {threshold:.6f}")
print(f"  Background flagged: {(bkg_scores > threshold).sum():,} / {len(bkg_scores):,}")
print(f"  Signal flagged:     {(sig_scores > threshold).sum():,} / {len(sig_scores):,}")

## Metrics

### Summary Metrics

ROC AUC, max SIC, and per-sample-type ROC AUC.

In [ ]:
from src.models.evaluation import compute_metrics, compute_roc_curve, compute_sic_curve

scores_df = build_scores_frame(
    scores, dm.predict_labels, dm.predict_origins, dm.predict_sample_types
)
metrics = compute_metrics(
    labels=dm.predict_labels,
    scores=scores,
    scores_df=scores_df,
)
metrics["threshold"] = threshold
metrics["n_background"] = int((dm.predict_labels == 0).sum())
metrics["n_signal"] = int((dm.predict_labels == 1).sum())

print(f"VAE ROC AUC: {metrics['roc_auc']:.4f}")
print(f"VAE max SIC: {metrics['max_sic']:.4f}")

### Reconstruction Error Distribution

Comparing the reconstruction error distributions of background and signal events.

In [ ]:
from src.models.plots import plot_reconstruction_error
from src.visualization.plots import save_figure

fig = plot_reconstruction_error(
    bkg_scores, sig_scores, threshold=threshold, title="VAE Reconstruction Error"
)
save_figure(fig, plots_dir / "reconstruction_error.png")
fig.show()

### ROC Curve

In [ ]:
from src.models.plots import plot_roc_curve

fpr, tpr, _ = compute_roc_curve(dm.predict_labels, scores)
fig = plot_roc_curve(fpr, tpr, auc=metrics["roc_auc"], title="VAE ROC Curve")
save_figure(fig, plots_dir / "roc_curve.png")
fig.show()

### SIC Curve

Significance Improvement Characteristic: signal efficiency divided by the square root of background efficiency.

In [ ]:
from src.models.plots import plot_sic_curve

sic = compute_sic_curve(fpr, tpr)
fig = plot_sic_curve(fpr, tpr, sic, title="VAE Significance Improvement")
save_figure(fig, plots_dir / "sic_curve.png")
fig.show()

### Per-Sample-Type ROC

ROC AUC per sample type. Signal: vs all background. Background types: one-vs-rest within background.

In [ ]:
from omegaconf import OmegaConf

from src.models.plots import plot_roc_per_sample_type

if metrics.get("roc_per_sample_type"):
    display_labels = OmegaConf.to_container(cfg.merge.display_labels, resolve=True)
    fig = plot_roc_per_sample_type(
        metrics["roc_per_sample_type"],
        title="VAE ROC AUC per Sample Type",
        display_labels=display_labels,
    )
    save_figure(fig, plots_dir / "roc_per_sample_type.png")
    fig.show()
    for st, auc in sorted(metrics["roc_per_sample_type"].items()):
        print(f"  {st}: {auc:.4f}")

### Per-Feature Importance

Mean squared reconstruction error per feature, identifying which features contribute most to the anomaly score.

In [ ]:
from src.models.plots import plot_per_feature_importance

feat_errors = per_feature_error(x_orig, x_hat).numpy()
mean_feat_errors = feat_errors.mean(axis=0)
fig = plot_per_feature_importance(
    mean_feat_errors, dm.feature_names_, title="VAE Per-Feature Reconstruction Error"
)
save_figure(fig, plots_dir / "per_feature_importance.png")
fig.show()

## Reconstruction Diagnostics

### Single Event Reconstruction

Bar chart comparing original vs reconstructed feature values for a single event.

In [ ]:
from src.models.plots import plot_reconstruction_performance

fig = plot_reconstruction_performance(
    x_orig[0].numpy(),
    x_hat[0].numpy(),
    dm.feature_names_,
    event_idx=0,
    title="Single Event Reconstruction (VAE)",
)
save_figure(fig, plots_dir / "single_event.png")
fig.show()

### Feature Histograms

Per-feature distributions of original vs reconstructed values across the predict set.

In [ ]:
from src.models.plots import plot_feature_histograms

fig = plot_feature_histograms(
    x_orig.numpy(),
    x_hat.numpy(),
    dm.feature_names_,
    title="Feature Distributions (VAE)",
)
save_figure(fig, plots_dir / "feature_histograms.png")
fig.show()

## VAE Latent Space Analysis

### Latent Dimension Histograms

Per-dimension histograms of the latent means coloured by background vs signal.

In [ ]:
from src.models.plots import plot_latent_histograms

fig = plot_latent_histograms(mu_np, labels=dm.predict_labels)
save_figure(fig, plots_dir / "latent_histograms.png")
fig.show()

### Latent Space 2D (t-SNE)

2D t-SNE embedding of the latent means coloured by background vs signal.

In [ ]:
from src.models.latent import compute_tsne
from src.models.plots import plot_latent_space_2d

max_tsne = 10_000
if len(mu_np) > max_tsne:
    rng = np.random.default_rng(cfg.seed)
    tsne_idx = rng.choice(len(mu_np), max_tsne, replace=False)
    tsne_mu = mu_np[tsne_idx]
    tsne_labels = dm.predict_labels[tsne_idx]
else:
    tsne_mu = mu_np
    tsne_labels = dm.predict_labels

embedding = compute_tsne(tsne_mu, seed=cfg.seed)
fig = plot_latent_space_2d(embedding, tsne_labels, method="t-SNE")
save_figure(fig, plots_dir / "tsne.png")
fig.show()

### Latent Pairplot

Pairwise scatter of the first latent dimensions (mu), coloured by background vs signal.

In [ ]:
from src.models.plots import plot_latent_pairplot

fig = plot_latent_pairplot(mu_np, labels=dm.predict_labels)
save_figure(fig, plots_dir / "latent_pairplot.png")
fig.show()

### Latent Mean Spread

Variance of `mu` per latent dimension. Dimensions with variance below 0.1 indicate potential posterior collapse.

In [ ]:
from src.models.plots import plot_latent_mean_spread

fig = plot_latent_mean_spread(mu_np)
save_figure(fig, plots_dir / "mu_spread.png")
fig.show()

### Log-Variance Spread

Mean `logvar` per latent dimension. Very negative values may indicate posterior collapse.

In [ ]:
from src.models.plots import plot_logvar_spread

fig = plot_logvar_spread(logvar_np)
save_figure(fig, plots_dir / "logvar_spread.png")
fig.show()

### Mu vs Logvar

Scatter plot of mean `mu` vs mean `logvar` per latent dimension.

In [ ]:
from src.models.plots import plot_mu_vs_logvar

fig = plot_mu_vs_logvar(mu_np, logvar_np)
save_figure(fig, plots_dir / "mu_vs_logvar.png")
fig.show()

### KL per Dimension

Mean KL divergence per latent dimension. Identifies which dimensions encode the most information.

In [ ]:
from src.models.latent import compute_kl_per_dimension
from src.models.plots import plot_kl_per_dimension

kl_per_dim = compute_kl_per_dimension(mu_np, logvar_np)
fig = plot_kl_per_dimension(kl_per_dim)
save_figure(fig, plots_dir / "kl_per_dim.png")
fig.show()

### Latent Mean Histograms

Per-dimension histograms of the latent mean `mu`.

In [ ]:
from src.models.plots import plot_latent_mean_histograms

fig = plot_latent_mean_histograms(mu_np)
save_figure(fig, plots_dir / "mu_histograms.png")
fig.show()

### Log-Variance Histograms

Per-dimension histograms of `logvar`.

In [ ]:
from src.models.plots import plot_logvar_histograms

fig = plot_logvar_histograms(logvar_np)
save_figure(fig, plots_dir / "logvar_histograms.png")
fig.show()

### Latent Mean Boxplot

Full distribution of the posterior mean per latent dimension. Boxes centred far from zero indicate the posterior is not aligned with the prior N(0, I).

In [ ]:
from src.models.plots import plot_latent_mean_boxplot

fig = plot_latent_mean_boxplot(mu_np)
save_figure(fig, plots_dir / "mu_boxplot.png")
fig.show()

### Log-Variance Boxplot

Full distribution of the posterior log-variance per latent dimension. Boxes well below zero indicate dimensions are more deterministic than the prior.

In [ ]:
from src.models.plots import plot_logvar_boxplot

fig = plot_logvar_boxplot(logvar_np)
save_figure(fig, plots_dir / "logvar_boxplot.png")
fig.show()

### Sampled vs Encoded Latent Space

Overlay of the reparameterised posterior `z = mu + eps * std` against samples from the prior N(0, I). 

In [ ]:
from src.models.plots import plot_sampled_latent_space

std_np = np.exp(0.5 * logvar_np)
z_encoded = mu_np + np.random.randn(*mu_np.shape) * std_np
z_sampled = np.random.randn(*mu_np.shape)

fig = plot_sampled_latent_space(z_sampled, z_encoded)
save_figure(fig, plots_dir / "sampled_latent.png")
fig.show()

## Serialization

### Scores DataFrame

Saving the anomaly scores to parquet.

In [ ]:
scores_path = dataframes_dir / "vae_scores.parquet"
scores_df.to_parquet(scores_path)
print(f"Saved scores: {scores_path}")
scores_df.head()

### Metrics JSON

Saving evaluation metrics.

In [ ]:
import json

metrics_path = dataframes_dir / "vae_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics: {metrics_path}")

### Finish WandB

Closing any active WandB run.

In [ ]:
import wandb

if wandb.run is not None:
    wandb.finish()